In [ ]:
!gdown 1LmC9qMMe1V6n21pkqZZ-4CLk5wiUfl6I

Downloading...
From: https://drive.google.com/uc?id=1LmC9qMMe1V6n21pkqZZ-4CLk5wiUfl6I
To: /content/df_train.csv
100% 286k/286k [00:00<00:00, 28.2MB/s]


In [ ]:
!gdown 1TeCL6hc2ltL8db09CeedC1tfEiDUbcxG

Downloading...
From: https://drive.google.com/uc?id=1TeCL6hc2ltL8db09CeedC1tfEiDUbcxG
To: /content/df_test.csv
100% 71.9k/71.9k [00:00<00:00, 59.2MB/s]


In [ ]:
import pandas as pd

df = pd.read_csv("df_train.csv")
df

,dwt_mean,dwt_max,dwt_min,dwt_median,dwt_q1,dwt_q3,dwt_iqr,mfcc_mean,mfcc_std,mfcc_max,...,rms_min,rms_median,rms_var,rms_skew,rms_kurtosis,rms_q1,rms_q3,rms_range,shannon_entropy,audio_type
0,1.206750e-08,0.013878,-0.014668,-1.500644e-11,-0.000028,0.000026,0.000055,-81.473040,295.34805,152.37076,...,9.717603e-08,0.000048,1.730892e-07,1.283449,0.308629,1.093666e-05,0.000524,0.001483,8.866710,myocardial
1,5.968118e-07,0.039521,-0.040875,1.670743e-11,-0.000016,0.000017,0.000033,-82.751785,298.57016,158.75189,...,1.647872e-07,0.000034,7.909553e-08,4.731361,22.492230,9.232333e-06,0.000076,0.001755,5.972518,myocardial
2,3.870791e-07,0.032473,-0.023627,7.594341e-12,-0.000017,0.000016,0.000033,-81.980400,297.24650,171.03497,...,1.124997e-07,0.000082,3.880997e-08,3.890662,20.913025,3.342536e-06,0.000215,0.001343,7.658798,myocardial
3,-4.941957e-07,0.131609,-0.118817,-2.006760e-11,-0.000014,0.000014,0.000028,-77.834270,280.35214,168.92433,...,9.554750e-08,0.000026,5.670202e-06,4.185934,17.443203,5.695775e-06,0.000075,0.015121,7.720282,myocardial
4,4.204849e-07,0.002665,-0.002124,-7.600626e-11,-0.000031,0.000029,0.000060,-82.015420,298.52830,97.61485,...,8.227275e-07,0.000036,3.165749e-09,0.953618,-0.323343,1.287166e-05,0.000091,0.000202,9.517833,myocardial
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
443,-1.585129e-08,0.011102,-0.009699,-2.031497e-09,-0.000019,0.000020,0.000039,-82.583570,297.38315,139.96802,...,1.878590e-06,0.000031,8.288504e-09,3.186870,11.402454,1.837468e-05,0.000060,0.000525,7.771392,normal
444,5.693161e-07,0.022825,-0.028649,-4.278836e-11,-0.000013,0.000013,0.000027,-83.153275,299.87430,200.29291,...,1.362070e-07,0.000027,1.346874e-07,3.889586,15.481110,1.185905e-05,0.000067,0.002202,5.310026,normal
445,4.123294e-09,0.060122,-0.060773,-4.331842e-10,-0.000004,0.000004,0.000008,-84.163050,296.94974,180.51411,...,7.595175e-07,0.000011,9.946041e-07,2.224082,3.260276,5.449809e-06,0.000059,0.003769,5.532661,normal
446,-3.339084e-08,0.089756,-0.102157,1.016094e-11,-0.000002,0.000002,0.000004,-82.164860,292.40936,203.16748,...,1.403314e-07,0.000011,1.463136e-06,5.434349,30.395962,8.566282e-07,0.000142,0.008902,6.255238,normal


In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dropout, Dense, BatchNormalization, Activation
from tensorflow.keras.optimizers import SGD
from sklearn.model_selection import KFold, train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix

In [ ]:
X = df.drop('audio_type', axis=1).values
y = df['audio_type'].values

# Encode categorical labels to numerical
from sklearn.preprocessing import LabelEncoder, StandardScaler
label_encoder = LabelEncoder()
y_numerical = label_encoder.fit_transform(y)

# One-hot encode numerical labels
y_one_hot = tf.keras.utils.to_categorical(y_numerical, num_classes=2)

# Normalize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Check for NaN values in features
if np.isnan(X_scaled).any():
    print("Warning: NaN values found in scaled features (X_scaled). This can cause 'nan' loss during training.")
else:
    print("No NaN values found in scaled features (X_scaled).")

No NaN values found in scaled features (X_scaled).


In [ ]:
X_train_full, X_test, y_train_full, y_test = train_test_split(X_scaled, y_one_hot, test_size=0.1, random_state=42)

X_train_full = X_train_full.reshape(X_train_full.shape[0], 51, 1)
X_test = X_test.reshape(X_test.shape[0], 51, 1)

In [ ]:
df_test = pd.read_csv('df_test.csv')

In [ ]:
X_test_data = df_test.drop('audio_type', axis=1).values
y_test_data = df_test['audio_type'].values

# Encode categorical labels to numerical using the fitted label_encoder
y_numerical_test = label_encoder.transform(y_test_data)

# One-hot encode numerical labels
y_one_hot_test = tf.keras.utils.to_categorical(y_numerical_test, num_classes=2)

# Normalize features using the fitted scaler
X_scaled_test = scaler.transform(X_test_data)

# Reshape input to (Samples, 51, 1) for 1D CNN
X_scaled_test = X_scaled_test.reshape(X_scaled_test.shape[0], 51, 1)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score, f1_score
import numpy as np
import pandas as pd # Import pandas for tabular output

# Prepare flattened training and test data for traditional ML models
# X_scaled is (Samples, 51, 1), reshape to (Samples, 51)
X_train_flat = X_scaled.reshape(X_scaled.shape[0], -1)
y_train_num = y_numerical

# X_scaled_test is (Samples, 51, 1), reshape to (Samples, 51)
X_test_flat = X_scaled_test.reshape(X_scaled_test.shape[0], -1)
y_test_num = y_numerical_test

all_run_results = {}

num_runs = 5

for run in range(num_runs):
    print(f"\n=======================================================")
    print(f"             Starting Run {run + 1}/{num_runs}              ")
    print(f"=======================================================")

    # Define the classifiers with varied random_state
    # Adding 'run' to random_state to introduce slight variations
    classifiers = {
        "Logistic Regression": LogisticRegression(random_state=42 + run, solver='liblinear'),
        "SVM (Linear)": SVC(kernel='linear', random_state=42 + run, probability=True),
        "SVM (RBF)": SVC(kernel='rbf', random_state=42 + run, probability=True),
        "Random Forest": RandomForestClassifier(random_state=42 + run),
        "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42 + run),
        "MLP": MLPClassifier(random_state=42 + run, max_iter=500, early_stopping=True, validation_fraction=0.1)
    }

    results = {}

    for name, clf in classifiers.items():
        print(f"\n--- Training and Evaluating {name} (Run {run + 1}) ---")

        # Train the model
        clf.fit(X_train_flat, y_train_num)

        # Make predictions
        y_pred = clf.predict(X_test_flat)
        y_pred_proba = clf.predict_proba(X_test_flat)[:, 1] # Probability for the positive class

        # Calculate metrics
        accuracy = accuracy_score(y_test_num, y_pred)
        cm = confusion_matrix(y_test_num, y_pred)
        tn, fp, fn, tp = cm.ravel()

        sensitivity = tp / (tp + fn) if (tp + fn) != 0 else 0
        specificity = tn / (tn + fp) if (tn + fp) != 0 else 0

        # Check if there's only one class in y_test_num for roc_auc_score
        if len(np.unique(y_test_num)) > 1:
            auroc = roc_auc_score(y_test_num, y_pred_proba)
        else:
            auroc = np.nan # Not applicable for single class
            print(f"Warning: Cannot compute AUROC for {name} as y_test_num contains only one class.")

        f1 = f1_score(y_test_num, y_pred)

        results[name] = {
            "Accuracy": accuracy,
            "Sensitivity": sensitivity,
            "Specificity": specificity,
            "AUROC": auroc,
            "F1-Score": f1,
            "Confusion Matrix": cm.tolist()
        }

        print(f"Accuracy: {accuracy*100:.2f}%")
        print(f"Sensitivity: {sensitivity*100:.2f}%")
        print(f"Specificity: {specificity*100:.2f}%")
        print(f"AUROC: {auroc*100:.2f}%")
        print(f"F1-Score: {f1:.2f}")
        print(f"Confusion Matrix:\n{cm}")

    all_run_results[f"Run_{run+1}"] = results

print("\n\n--- Detailed Results of All Classification Runs ---")

# Prepare data for tabular display showing all runs individually
detailed_table_data = []
for run_idx in range(num_runs):
    run_name = f"Run_{run_idx+1}"
    for classifier_name, metrics in all_run_results[run_name].items():
        detailed_table_data.append({
            "Classifier": classifier_name,
            "Run": run_idx + 1,
            "Accuracy": metrics["Accuracy"],
            "Sensitivity": metrics["Sensitivity"],
            "Specificity": metrics["Specificity"],
            "AUROC": metrics["AUROC"],
            "F1-Score": metrics["F1-Score"]
        })

detailed_df = pd.DataFrame(detailed_table_data)

# Format percentages and F1-Score for better readability
for col in detailed_df.columns:
    if "Accuracy" in col or "Sensitivity" in col or "Specificity" in col or "AUROC" in col:
        detailed_df[col] = detailed_df[col].apply(lambda x: f"{x*100:.2f}%" if not pd.isna(x) else "N/A")
    elif "F1-Score" in col:
        detailed_df[col] = detailed_df[col].apply(lambda x: f"{x:.2f}" if not pd.isna(x) else "N/A")

print(detailed_df.to_string())



             Starting Run 1/5              

--- Training and Evaluating Logistic Regression (Run 1) ---
Accuracy: 82.14%
Sensitivity: 82.14%
Specificity: 82.14%
AUROC: 90.59%
F1-Score: 0.82
Confusion Matrix:
[[46 10]
 [10 46]]

--- Training and Evaluating SVM (Linear) (Run 1) ---
Accuracy: 82.14%
Sensitivity: 82.14%
Specificity: 82.14%
AUROC: 90.72%
F1-Score: 0.82
Confusion Matrix:
[[46 10]
 [10 46]]

--- Training and Evaluating SVM (RBF) (Run 1) ---
Accuracy: 83.04%
Sensitivity: 82.14%
Specificity: 83.93%
AUROC: 89.92%
F1-Score: 0.83
Confusion Matrix:
[[47  9]
 [10 46]]

--- Training and Evaluating Random Forest (Run 1) ---
Accuracy: 83.04%
Sensitivity: 76.79%
Specificity: 89.29%
AUROC: 90.10%
F1-Score: 0.82
Confusion Matrix:
[[50  6]
 [13 43]]

--- Training and Evaluating XGBoost (Run 1) ---
Accuracy: 83.93%
Sensitivity: 87.50%
Specificity: 80.36%
AUROC: 89.73%
F1-Score: 0.84
Confusion Matrix:
[[45 11]
 [ 7 49]]

--- Training and Evaluating MLP (Run 1) ---
Accuracy: 75.89%
Sensitiv